In [9]:
import pdfplumber
import pandas as pd
from datetime import datetime
import re

electric_details = {}
gas_details = {}

pdf_path = "BGE/20250716.pdf"
# pdf_path = "BGE/20250815.pdf"
pdf_path = "BGE/20250616.pdf"

# Function returns standard date output; Jun22,2025 -> 2025-06-22
def format_date(input_date:str):
    # Expects abbreviated month name
    date_obj = datetime.strptime(input_date, "%b%d,%Y")
    format_date = date_obj.strftime("%Y-%m-%d")
    return format_date

def extract_electric(input_table:list, output_dict:dict):
    for item in input_table:
        row = item
        row = (item[0] or "").splitlines()
        # print(row)
        for entry in row:
            # Total Energy Used
            if "Current - Previous" in entry:
                output_dict["total_kWh"] = float(entry.split("= ")[1])

            # Billing Periods
            if "BillingPeriod" in entry:
                date_paentryern = r'[A-Z][a-z]{2}\d{1,2},\d{4}'
                dates = re.findall(date_paentryern, entry)
                output_dict["billing_period_start"] = format_date(dates[0])
                output_dict["billing_period_end"] = format_date(dates[1])

            if "ELECTRICSUPPLY" in entry:
                indices = [i for i, target in enumerate(row) if "ELECTRICSUPPLY" in target or "BGEELECTRICDELIVERY" in target]
                print(row)
                number_rates = indices[1] - indices[0] - 1
                for i in range(0,number_rates):
                    # Skip first entry b/c it's an anchor
                    search_idx = row[i+1]
                    rate_key = f"electric_supply_{i}_rate"
                    rate_energy_key = f"electric_supply_{i}_energy"
                    rate_price_key = f"electric_supply_{i}_price"

                    # Skips first entry
                    print(search_idx)
                    # Optional integer or floating point extraction
                    output_dict[rate_energy_key] = float(re.search(r"(\d+(?:\.\d+)?)kWh", search_idx).group(1))
                    output_dict[rate_key] = float(re.search(r"x\s*(\.\d+)", search_idx).group(1))
                    output_dict[rate_price_key] = search_idx.split(" ")[-1]
            
            # BGE Electric Delivery - assumes all 
            ## Customer Charge
            if "CustomerCharge" in entry:
                output_dict["delivery_chg_cust"] = entry.split(" ")[1]
            ## EmPower MD Charge
            if "EmPowerMDChg" in entry:
                output_dict["delivery_chg_empower_md"] = float(re.search(r"x\s*(\.\d+)", entry).group(1))
            ## Distribution Charge
            if "DistributionChg" in entry:
                output_dict["delivery_chg_distribution"] = float(re.search(r"x\s*(\.\d+)", entry).group(1))


    # output_dict["total_kWh"] = input_table[3][1].splitlines()[0]

with pdfplumber.open(pdf_path) as pdf:
    first_page = pdf.pages[1]
    table = first_page.extract_tables()
    electric_table = table[0]
    gas_table = table[1]
    extract_electric(electric_table, electric_details)
    print(electric_details)
    # df = pd.DataFrame(table[1:], columns=table[0]) 



['ELECTRICSUPPLY $75.92', 'BGE 638kWh x .11899 75.92', 'BGEELECTRICDELIVERY $48.91', 'CustomerCharge 9.65', 'EmPowerMDChg 638kWh x .01028 6.56', 'DistributionChg 638kWh x .05125 32.70', 'TAXES&FEES $0.82', 'MDUniversalSvcProg 0.32', 'EnvirSrchg 638kWh x .00015 0.10', 'FranchiseTax 638kWh x .00062 0.40']
BGE 638kWh x .11899 75.92
{'billing_period_start': '2025-04-23', 'billing_period_end': '2025-05-22', 'total_kWh': 638.0, 'electric_supply_0_energy': 638.0, 'electric_supply_0_rate': 0.11899, 'electric_supply_0_price': '75.92', 'delivery_chg_cust': '9.65', 'delivery_chg_empower_md': 0.01028, 'delivery_chg_distribution': 0.05125}
